# starfold quickstart

This notebook runs `starfold`'s full methodology on a synthetic
Gaussian-mixture dataset. You should be able to execute it
top-to-bottom with `pip install starfold` and nothing else.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_blobs

import starfold as sf

## 1. Data

A 5-D, 4-blob dataset. `starfold` takes any `(n_samples, n_features)`
float array, so the origin of `X` is irrelevant to the tool.

In [ ]:
X, y_true = make_blobs(
    n_samples=600, n_features=5, centers=4,
    cluster_std=1.5, random_state=0,
)
X.shape, y_true.shape

## 2. Pipeline

`UnsupervisedPipeline` runs StandardScaler → UMAP → Optuna-HDBSCAN
→ trustworthiness (+ optional noise baseline). Here we skip the
(expensive) noise baseline for the quickstart; run with
`skip_noise_baseline=False` to get cluster-significance flags.

In [ ]:
pipeline = sf.UnsupervisedPipeline(
    umap_kwargs={"n_epochs": 2000, "n_neighbors": 30},
    hdbscan_optuna_trials=40,
    mcs_range=(20, 120),
    ms_range=(5, 25),
    skip_noise_baseline=True,
    random_state=0,
)
result = pipeline.fit(X)
print(result.summary())

## 3. Plots

The `plot_*` helpers return matplotlib Axes so you can compose panels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sf.plot_embedding(result.embedding, result.labels, ax=axes[0])
axes[0].set_title(f"{result.n_clusters} clusters, T = {result.trustworthiness:.3f}")
sf.plot_optuna_history(result.search.study, ax=axes[1])
plt.show()

In [ ]:
scores = sf.trustworthiness_curve(
    pipeline.scaler.fit_transform(X) if hasattr(pipeline, 'scaler') else X,
    result.embedding,
    k_values=(5, 10, 15, 25, 50),
)
sf.plot_trustworthiness_curve(scores)
plt.show()

## 4. Compare dimensionality-reduction methods

For diagnostic purposes, PCA, t-SNE, and UMAP are all available.

In [ ]:
emb_pca = sf.run_pca(X, random_state=0)
emb_tsne = sf.run_tsne(X, n_iter=1000, random_state=0)
emb_umap = result.embedding
fig, _ = sf.plot_embedding_comparison(
    {"PCA": emb_pca, "t-SNE": emb_tsne, "UMAP": emb_umap},
    result.labels,
    figsize=(15, 4.5),
)
plt.show()

## 5. Two-run workflow (optional)

The paper reruns the pipeline on each top-level component separately.
`starfold` does not bake this in; you filter the labels and call
`fit` again:

```python
mask = result.labels == 0
sub_result = pipeline.fit(X[mask])
```